In [3]:
import decimal
import json
import logging
from typing import Any, Dict, List
from boto3.dynamodb.types import TypeDeserializer

import boto3
from botocore.exceptions import ClientError
from IPython.display import HTML

In [4]:
region="us-east-1"

In [5]:
def create_tables_from_json(json_file):
        import json
        dynamodb = boto3.client("dynamodb", region_name=region)
        try:
            with open(json_file, "r") as f:
                config = json.load(f)

            for table in config["tables"]:
                print(f"Creating table: {table['table_name']}")

                dynamodb.create_table(
                    TableName=table["table_name"],
                    KeySchema=table["kwargs"]["KeySchema"],
                    AttributeDefinitions=table["kwargs"]["AttributeDefinitions"],
                    ProvisionedThroughput=table["kwargs"]["ProvisionedThroughput"]
                )

                waiter = dynamodb.get_waiter("table_exists")
                waiter.wait(TableName=table["table_name"])

                print(f"Table created: {table['table_name']}")

        except ClientError as e:
            print("Error:", e)

In [6]:
response = create_tables_from_json('table_property.json') 
print(response)

Creating table: ProductCatalog
Table created: ProductCatalog
Creating table: Forum
Table created: Forum
Creating table: Thread
Table created: Thread
Creating table: Reply
Table created: Reply
None


In [7]:
def read_data(file_path: str) -> Dict[str, Any]:
    with open(file_path, "r") as json_file:
        items = json.load(json_file)
    return items

In [8]:
read_data("data/Forum.json")

{'Forum': [{'PutRequest': {'Item': {'Name': {'S': 'Amazon DynamoDB'},
     'Category': {'S': 'Amazon Web Services'},
     'Threads': {'N': '2'},
     'Messages': {'N': '4'},
     'Views': {'N': '1000'}}}},
  {'PutRequest': {'Item': {'Name': {'S': 'Amazon S3'},
     'Category': {'S': 'Amazon Web Services'}}}}]}

In [9]:
def load_data_from_json(config_file):
       # try:
            client = boto3.client("dynamodb")
            

            with open(config_file, "r") as f:
                config = json.load(f)
            
            if isinstance(config, dict):
                items = list(config.values())[0]   # get the first list
            else:
                items = config

            for table_config in config["tables"]:
                table_name = table_config["table_name"]
                file_path = table_config["file"]

                print(f"\nLoading data into {table_name} from {file_path}")
                with open(file_path, "r") as json_file:
                    items = json.load(json_file)
                    item=items[table_name.split('-')[-1]]
                    #print(f'{item}')
                
                    client.put_item(TableName=table_name, Item=item[0]['PutRequest']['Item'])
                   # print(f"{item[0]['PutRequest']['Item']} in {table_name}")





                

       # except ClientError as e:
        #    print("AWS Error:", e)
       # except Exception as e:
         #   print("Error:", e)

In [28]:
load_data_from_json("data_config.json")


Loading data into mycourse-ProductCatalog from data/ProductCatalog.json

Loading data into mycourse-Forum from data/Forum.json

Loading data into mycourse-Thread from data/Thread.json

Loading data into mycourse-Reply from data/Reply.json


In [10]:
def batch_write_item_db(config_file):
       # try:
            
            client = boto3.client("dynamodb")

            with open(config_file, "r") as f:
                config = json.load(f)
            

            for table_config in config["tables"]:
                table_name = table_config["table_name"]
                file_path = table_config["file"]

                print(f"\nLoading data into {table_name} from {file_path}")
                with open(file_path, "r") as json_file:
                    items = json.load(json_file)
                   
                
                    client.batch_write_item( RequestItems=items)
                    print(f"{items} in {table_name}")





                

       # except ClientError as e:
        #    print("AWS Error:", e)
       # except Exception as e:
         #   print("Error:", e)

In [11]:
batch_write_item_db("data_config.json")


Loading data into ProductCatalog from data/ProductCatalog.json
{'ProductCatalog': [{'PutRequest': {'Item': {'Id': {'N': '101'}, 'Title': {'S': 'Book 101 Title'}, 'ISBN': {'S': '111-1111111111'}, 'Authors': {'L': [{'S': 'Author1'}]}, 'Price': {'N': '2'}, 'Dimensions': {'S': '8.5 x 11.0 x 0.5'}, 'PageCount': {'N': '500'}, 'InPublication': {'BOOL': True}, 'ProductCategory': {'S': 'Book'}}}}, {'PutRequest': {'Item': {'Id': {'N': '102'}, 'Title': {'S': 'Book 102 Title'}, 'ISBN': {'S': '222-2222222222'}, 'Authors': {'L': [{'S': 'Author1'}, {'S': 'Author2'}]}, 'Price': {'N': '20'}, 'Dimensions': {'S': '8.5 x 11.0 x 0.8'}, 'PageCount': {'N': '600'}, 'InPublication': {'BOOL': True}, 'ProductCategory': {'S': 'Book'}}}}, {'PutRequest': {'Item': {'Id': {'N': '103'}, 'Title': {'S': 'Book 103 Title'}, 'ISBN': {'S': '333-3333333333'}, 'Authors': {'L': [{'S': 'Author1'}, {'S': 'Author2'}]}, 'Price': {'N': '2000'}, 'Dimensions': {'S': '8.5 x 11.0 x 1.5'}, 'PageCount': {'N': '600'}, 'InPublication': {'

In [2]:
def scan_db(table_name: str, **kwargs):
    ### START CODE HERE ### (~ 2 lines of code)
    client = boto3.client("dynamodb")
    response = client.scan(TableName=table_name, **kwargs)
    ### END CODE HERE ###
    
    return response

In [3]:
response = scan_db('Forum')
print(f"Queried data for table :\n{response}")

Queried data for table :
{'Items': [{'Category': {'S': 'Amazon Web Services'}, 'Name': {'S': 'Amazon S3'}}, {'Threads': {'N': '2'}, 'Category': {'S': 'Amazon Web Services'}, 'Messages': {'N': '4'}, 'Views': {'N': '1000'}, 'Name': {'S': 'Amazon DynamoDB'}}], 'Count': 2, 'ScannedCount': 2, 'ResponseMetadata': {'RequestId': '6IBL8MNOUK2VJ1719PL0OCVQMNVV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Sun, 26 Apr 2026 17:24:41 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '238', 'connection': 'keep-alive', 'x-amzn-requestid': '6IBL8MNOUK2VJ1719PL0OCVQMNVV4KQNSO5AEMVJF66Q9ASUAAJG', 'x-amz-crc32': '4247158843'}, 'RetryAttempts': 0}}


In [4]:
def data_deserializer(data: Dict[str, Any]):
    boto3.resource("dynamodb")

    deserializer = boto3.dynamodb.types.TypeDeserializer()

    deserialized_data = {
        k: (
            float(deserializer.deserialize(v))
            if isinstance(deserializer.deserialize(v), decimal.Decimal)
            else deserializer.deserialize(v)
        )
        for k, v in data.items()
    }

    return deserialized_data

In [5]:
for item in response['Items']:
    print(f"DynamoDB returned Marshal JSON:\n{item}")
    print(f"Deserialized python dictionary:\n {data_deserializer(item)}")

DynamoDB returned Marshal JSON:
{'Category': {'S': 'Amazon Web Services'}, 'Name': {'S': 'Amazon S3'}}
Deserialized python dictionary:
 {'Category': 'Amazon Web Services', 'Name': 'Amazon S3'}
DynamoDB returned Marshal JSON:
{'Threads': {'N': '2'}, 'Category': {'S': 'Amazon Web Services'}, 'Messages': {'N': '4'}, 'Views': {'N': '1000'}, 'Name': {'S': 'Amazon DynamoDB'}}
Deserialized python dictionary:
 {'Threads': 2.0, 'Category': 'Amazon Web Services', 'Messages': 4.0, 'Views': 1000.0, 'Name': 'Amazon DynamoDB'}


In [11]:


def scan_table( table_name):
    client = boto3.client("dynamodb")
    deserializer = TypeDeserializer()
    items = []
    last_evaluated_key = None

    while True:
        if last_evaluated_key:
            response = client.scan(
                TableName=table_name,
                ExclusiveStartKey=last_evaluated_key
            )
        else:
            response = client.scan(TableName=table_name)

        batch = response.get("Items", [])

        for item in batch:
            items.append({
                k: deserializer.deserialize(v)
                for k, v in item.items()
            })

        last_evaluated_key = response.get("LastEvaluatedKey")

        if not last_evaluated_key:
            break

    return items

In [12]:
scan_table('Forum')

[{'Category': 'Amazon Web Services', 'Name': 'Amazon S3'},
 {'Threads': Decimal('2'),
  'Category': 'Amazon Web Services',
  'Messages': Decimal('4'),
  'Views': Decimal('1000'),
  'Name': 'Amazon DynamoDB'}]

In [14]:
def get_item_db(table_name, key: Dict[str, Any], **kwargs):
    client = boto3.client("dynamodb")

    try:
        ### START CODE HERE ### (~ 1 line of code)
        response = client.get_item(TableName=table_name, Key=key, **kwargs)
        ### END CODE HERE ###
        
    except ClientError as e:
        error = e.response.get("Error", {})
        logging.error(
            f"Failed to query DynamoDB. Error: {error.get('Message')}"
        )
        response = {}
    
    return response

In [16]:
response = get_item_db(table_name='ProductCatalog', 
                    key={'Id': {'N': '101'}})
print(response)

{'Item': {'Title': {'S': 'Book 101 Title'}, 'InPublication': {'BOOL': True}, 'PageCount': {'N': '500'}, 'Dimensions': {'S': '8.5 x 11.0 x 0.5'}, 'ISBN': {'S': '111-1111111111'}, 'Authors': {'L': [{'S': 'Author1'}]}, 'Price': {'N': '2'}, 'ProductCategory': {'S': 'Book'}, 'Id': {'N': '101'}}, 'ResponseMetadata': {'RequestId': 'F60740J5COC700QJQ3ON6HJ6NVVV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Sun, 26 Apr 2026 17:48:33 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '263', 'connection': 'keep-alive', 'x-amzn-requestid': 'F60740J5COC700QJQ3ON6HJ6NVVV4KQNSO5AEMVJF66Q9ASUAAJG', 'x-amz-crc32': '3181387427'}, 'RetryAttempts': 0}}


In [ ]:
def query_db(table_name: str,**kwargs,):
        client = boto3.client("dynamodb")

        try:
            response = client.query(
                TableName=table_name,
                **kwargs,
            )
            logging.info(f"Response {response}")
        except ClientError as e:
            error = e.response.get("Error", {})
            logging.error(
                f"Failed to query DynamoDB. Error: {error.get('Message')}"
            )
            raise
        else:
            logging.info(f"Query result {response.get('Items', {})}")
            return response

In [33]:
kwargs = {'ReturnConsumedCapacity': 'TOTAL', 
          'KeyConditionExpression': 'Id = :Id and ReplyDateTime > :ts',
          'ExpressionAttributeValues': {':Id': {'S': 'Amazon DynamoDB#DynamoDB Thread 1'}, 
                               ':ts' : {'S':"2015-09-21"}
                               }
          }

query_db('Reply', **kwargs)

{'Items': [{'Message': {'S': 'DynamoDB Thread 1 Reply 2 text'},
   'PostedBy': {'S': 'User B'},
   'Id': {'S': 'Amazon DynamoDB#DynamoDB Thread 1'},
   'ReplyDateTime': {'S': '2015-09-22T19:58:22.947Z'}}],
 'Count': 1,
 'ScannedCount': 1,
 'ConsumedCapacity': {'TableName': 'Reply', 'CapacityUnits': 0.5},
 'ResponseMetadata': {'RequestId': 'C97PQOJOD7IT89R5FNFIO81PTFVV4KQNSO5AEMVJF66Q9ASUAAJG',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'server': 'Server',
   'date': 'Mon, 27 Apr 2026 11:35:47 GMT',
   'content-type': 'application/x-amz-json-1.0',
   'content-length': '272',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'C97PQOJOD7IT89R5FNFIO81PTFVV4KQNSO5AEMVJF66Q9ASUAAJG',
   'x-amz-crc32': '3640125574'},
  'RetryAttempts': 0}}

In [40]:
import boto3
import logging
from botocore.exceptions import ClientError

def query_db(table_name: str, **kwargs):
    client = boto3.client("dynamodb")

    items = []
    last_evaluated_key = None

    try:
        while True:
            if last_evaluated_key:
                kwargs["ExclusiveStartKey"] = last_evaluated_key

            response = client.query(
                TableName=table_name,
                **kwargs,
            )
            logging.info(f"Response {response}")
            items.extend(response.get("Items", []))
            logging.info(f"Response {response}")
            last_evaluated_key = response.get("LastEvaluatedKey")
            if not last_evaluated_key:
                break
        

        print(f"Total items retrieved: {len(items)}")
        print(f'{response}') 
        return items

    except ClientError as e:
        error = e.response.get("Error", {})
        logging.error(f"Failed to query DynamoDB: {error.get('Message')}")
        raise
    else:
            logging.info(f"Query result {response.get('Items', {})}")
            return response

In [41]:
kwargs = {'ReturnConsumedCapacity': 'TOTAL', 
          'KeyConditionExpression': 'Id = :Id and ReplyDateTime > :ts',
          'ExpressionAttributeValues': {':Id': {'S': 'Amazon DynamoDB#DynamoDB Thread 1'}, 
                               ':ts' : {'S':"2015-09-21"}
                               }
          }

query_db('Reply', **kwargs)

Total items retrieved: 1
{'Items': [{'Message': {'S': 'DynamoDB Thread 1 Reply 2 text'}, 'PostedBy': {'S': 'User B'}, 'Id': {'S': 'Amazon DynamoDB#DynamoDB Thread 1'}, 'ReplyDateTime': {'S': '2015-09-22T19:58:22.947Z'}}], 'Count': 1, 'ScannedCount': 1, 'ConsumedCapacity': {'TableName': 'Reply', 'CapacityUnits': 0.5}, 'ResponseMetadata': {'RequestId': 'H8HMA9V034HBEMT3B53KOEOJNJVV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Mon, 27 Apr 2026 11:54:03 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '272', 'connection': 'keep-alive', 'x-amzn-requestid': 'H8HMA9V034HBEMT3B53KOEOJNJVV4KQNSO5AEMVJF66Q9ASUAAJG', 'x-amz-crc32': '3640125574'}, 'RetryAttempts': 0}}


[{'Message': {'S': 'DynamoDB Thread 1 Reply 2 text'},
  'PostedBy': {'S': 'User B'},
  'Id': {'S': 'Amazon DynamoDB#DynamoDB Thread 1'},
  'ReplyDateTime': {'S': '2015-09-22T19:58:22.947Z'}}]

In [42]:
def update_item_db(table_name: str, key: Dict[str, Any], **kwargs):
    client = boto3.client("dynamodb")

    response = client.update_item(
        TableName=table_name, Key=key, ReturnValues="UPDATED_NEW", **kwargs
    )

    return response

In [43]:
kwargs= {    
    'UpdateExpression': 'SET Messages = :newMessages',
    'ConditionExpression': 'Messages = :oldMessages',
    'ExpressionAttributeValues': {
        ":oldMessages" : {"N": "4"},
        ":newMessages" : {"N": "5"}
    }
}
response = update_item_db(table_name='Forum', key={'Name' : {'S': 'Amazon DynamoDB'}}, **kwargs)
print(response)

{'Attributes': {'Messages': {'N': '5'}}, 'ResponseMetadata': {'RequestId': 'QCDLA0KUM8IGT413GP5T46MJ8FVV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Mon, 27 Apr 2026 12:01:43 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '37', 'connection': 'keep-alive', 'x-amzn-requestid': 'QCDLA0KUM8IGT413GP5T46MJ8FVV4KQNSO5AEMVJF66Q9ASUAAJG', 'x-amz-crc32': '1508008640'}, 'RetryAttempts': 0}}


In [12]:
def delete_item_db(table_name: str, key: dict[str, Any], **kwargs):
    ### START CODE HERE ### (~ 2 lines of code)
    client = boto3.client("dynamodb")
    response = client.delete_item(TableName=table_name, Key=key, **kwargs)
    ### END CODE HERE ###
    
    logging.info(f"response {response}")

In [45]:
key = {"Id" : {"S": "Amazon DynamoDB#DynamoDB Thread 2"},
       "ReplyDateTime" : {"S": "2021-04-27T17:47:30Z"}
       }

delete_item_db(table_name='Reply', key=key)

In [ ]:
def transact_write_items_db(transaction_items: List[Dict[str, Any]], **kwargs):
    ### START CODE HERE ### (~ 2 lines of code)
    client = boto3.client("dynamodb")
    response = client.transact_write_items(TransactItems=transaction_items, **kwargs)
    ### END CODE HERE ###

    return response

In [14]:
def delete_table( table_name):
        client = boto3.client("dynamodb")
        try:
            client.delete_table(TableName=table_name)
            print(f"Deleted table: {table_name}")
        except ClientError as e:
            print("Error:", e)

In [15]:
for i in ['Forum','ProductCatalog','Reply','Thread']:
    delete_table(i)

Deleted table: Forum
Deleted table: ProductCatalog
Deleted table: Reply
Deleted table: Thread
